In [ ]:
from pathlib import Path

print("All top-level under /kaggle/input:")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(f"  {p}")
    if p.is_dir():
        for sub in sorted(p.iterdir())[:5]:
            print(f"    {sub.name}")

print("\nSearching anywhere for 'tracks' folder:")
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if d == "tracks":
            full = Path(root) / d
            n = sum(1 for _ in full.iterdir())
            print(f"  Found: {full}  ({n} items)")
    # Don't descend into massive frames folder
    if 'frames' in dirs:
        dirs.remove('frames')

In [ ]:
"""
Session 2, Full DoTA: features + labels + v4-clean + Stage-2 + STAUC.

Hardcoded paths:
  TRACKS: /kaggle/input/datasets/antonygerold6/dota-tracks-v4/tracks  (4880)
  ANN:    /kaggle/input/datasets/uom200399g/dota-dataset/dota-anno/annotations  (4677)
  FRAMES: not used (we skip Farneback optical flow to save time; costs ~0.002 STAUC)

CPU-only. Expected runtime 3-5 hours. Checkpointed: if session dies,
features_v3_full.parquet + other intermediates persist for rerun.
"""
import os, json, sys, time, copy
from pathlib import Path

print("Installing lightgbm + pyarrow...")
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"],
                   capture_output=True, text=True)
if r.returncode != 0: print(r.stderr); sys.exit(1)
print("OK")

import numpy as np
import pandas as pd
import torch
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.utils.extmath import stable_cumsum
from sklearn import metrics as sk_metrics
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

TRACKS_DIR = Path("/kaggle/input/datasets/antonygerold6/dota-tracks-v4/tracks")
ANN_DIR    = Path("/kaggle/input/datasets/uom200399g/dota-dataset/dota-anno/annotations")
OUT_DIR    = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
print("\n" + "="*72); print("Stage 1: Load annotations + cross-check"); print("="*72)
# ============================================================

track_files = sorted(TRACKS_DIR.glob("*.json"))
print(f"Track files: {len(track_files)}")

annotations = {}
for jf in ANN_DIR.glob("*.json"):
    a = json.load(open(jf))
    annotations[a['video_name']] = a
print(f"Annotations: {len(annotations)}")

track_names = {tp.stem for tp in track_files}
ann_names = set(annotations.keys())
eval_videos = sorted(track_names & ann_names)
print(f"Evaluation videos (tracks ∩ annotations): {len(eval_videos)}")

# ============================================================
print("\n" + "="*72); print("Stage 2: Base + interaction + temporal features"); print("="*72)
# ============================================================
FRAME_W, FRAME_H = 1280, 720
CACHE_V3 = OUT_DIR / "features_v3_full.parquet"

if CACHE_V3.exists():
    print(f"Loading cached: {CACHE_V3}")
    df = pd.read_parquet(CACHE_V3)
else:
    t0 = time.time()
    rows = []
    for vi, vid in enumerate(eval_videos):
        tp = TRACKS_DIR / f"{vid}.json"
        if not tp.exists(): continue
        tracks = json.load(open(tp))
        ann = annotations[vid]
        anom_s, anom_e = ann['anomaly_start'], ann['anomaly_end']

        track_seq = {}
        for f in tracks:
            for d in f.get('detections', []):
                track_seq.setdefault(d['track_id'], []).append(
                    (f['frame'], d['bbox'], d.get('class', -1), d.get('conf', 1.0)))

        for tid, seq in track_seq.items():
            seq = sorted(seq, key=lambda x: x[0])
            first_seen = seq[0][0]
            for i, (fid, bbox, cls, conf) in enumerate(seq):
                x1, y1, x2, y2 = bbox
                cx = (x1 + x2) / 2 / FRAME_W
                cy = (y1 + y2) / 2 / FRAME_H
                w = (x2 - x1) / FRAME_W
                h = (y2 - y1) / FRAME_H
                area_norm = w * h
                aspect = w / max(h, 1e-6)

                if i > 0:
                    p_fid, p_bbox, _, _ = seq[i-1]
                    p_cx = (p_bbox[0]+p_bbox[2])/2/FRAME_W
                    p_cy = (p_bbox[1]+p_bbox[3])/2/FRAME_H
                    dt = max(1, fid - p_fid)
                    vx = (cx - p_cx) / dt
                    vy = (cy - p_cy) / dt
                    p_area = ((p_bbox[2]-p_bbox[0])/FRAME_W) * ((p_bbox[3]-p_bbox[1])/FRAME_H)
                    area_change = (area_norm - p_area) / max(p_area, 1e-6)
                else:
                    vx = vy = area_change = 0

                speed = np.sqrt(vx*vx + vy*vy)
                direction = np.arctan2(vy, vx) if speed > 1e-6 else 0

                if i > 1:
                    pp_bbox = seq[i-2][1]; pp_fid = seq[i-2][0]
                    pp_cx = (pp_bbox[0]+pp_bbox[2])/2/FRAME_W
                    pp_cy = (pp_bbox[1]+pp_bbox[3])/2/FRAME_H
                    p_cx2 = (seq[i-1][1][0]+seq[i-1][1][2])/2/FRAME_W
                    p_cy2 = (seq[i-1][1][1]+seq[i-1][1][3])/2/FRAME_H
                    p_vx = (p_cx2 - pp_cx) / max(1, seq[i-1][0]-pp_fid)
                    p_vy = (p_cy2 - pp_cy) / max(1, seq[i-1][0]-pp_fid)
                    ax = vx - p_vx; ay = vy - p_vy
                    jerk = np.sqrt(ax*ax + ay*ay) if i > 2 else 0
                else:
                    ax = ay = jerk = 0

                rows.append({
                    'video': vid, 'frame': fid, 'track_id': tid,
                    'cx': cx, 'cy': cy, 'norm_x': cx, 'norm_bottom_y': y2/FRAME_H,
                    'area_norm': area_norm, 'aspect_ratio': aspect,
                    'area_change_rate': area_change,
                    'vx_norm': vx, 'vy_norm': vy, 'speed_norm': speed, 'direction_rad': direction,
                    'v4_flow_decoupled_score': 0.0,
                    'ax_norm': ax, 'ay_norm': ay, 'jerk_norm': jerk,
                    'track_age': fid - first_seen,
                    'track_stability': 1.0 if i > 5 else 0.5,
                    'yolo_class': cls, 'conf': conf,
                    'is_anomalous_orig': 1 if anom_s <= fid <= anom_e else 0,
                })
        if (vi + 1) % 200 == 0 or vi == 0:
            print(f"  base [{vi+1}/{len(eval_videos)}]  {(time.time()-t0)/60:.1f}m")

    df = pd.DataFrame(rows).sort_values(['video','track_id','frame']).reset_index(drop=True)
    print(f"Base df: {df.shape}  elapsed {(time.time()-t0)/60:.1f}m")

    # ---- Interaction + TTC ----
    print("\n  Computing interaction + TTC (~20-40m)...")
    t1 = time.time()
    for c in ['nn_dist_min','nn_dist_mean','nn_closing_max','nn_closing_mean',
              'nn_iou_max','nn_relative_speed_max','n_others_in_frame',
              'ttc_frames','inverse_ttc']:
        df[c] = 0.0
    df['nn_dist_min'] = 1.5; df['nn_dist_mean'] = 1.5
    K = 3
    groups = df.groupby(['video','frame'], sort=False)
    n_groups = groups.ngroups
    for gi, ((vid, fid), group) in enumerate(groups):
        positions = group.index.values
        n = len(positions)
        area_change = group['area_change_rate'].values.astype(np.float32)
        inv_ttc = area_change / 2.0
        ttc = np.where(np.abs(inv_ttc) > 1e-4, 1.0/np.where(inv_ttc==0, 1e-9, inv_ttc), 1000.0)
        df.loc[positions, 'ttc_frames'] = np.clip(ttc, -1000, 1000)
        df.loc[positions, 'inverse_ttc'] = np.clip(inv_ttc, -1, 1)
        df.loc[positions, 'n_others_in_frame'] = n - 1
        if n < 2: continue
        cx = group['cx'].values.astype(np.float32)
        cy = group['cy'].values.astype(np.float32)
        vx = group['vx_norm'].values.astype(np.float32)
        vy = group['vy_norm'].values.astype(np.float32)
        area = group['area_norm'].values.astype(np.float32)
        aspect = group['aspect_ratio'].values.astype(np.float32)
        h_norm = np.sqrt(np.maximum(area, 1e-12) / np.maximum(aspect, 1e-3))
        w_norm = aspect * h_norm
        x1 = cx - w_norm/2; x2 = cx + w_norm/2
        y1 = cy - h_norm/2; y2 = cy + h_norm/2
        dx = cx[None,:]-cx[:,None]; dy = cy[None,:]-cy[:,None]
        dist = np.sqrt(dx*dx + dy*dy); np.fill_diagonal(dist, np.inf)
        ds = np.maximum(dist, 1e-6)
        ux = dx/ds; uy = dy/ds
        rvx = vx[None,:]-vx[:,None]; rvy = vy[None,:]-vy[:,None]
        closing = -(rvx*ux + rvy*uy); np.fill_diagonal(closing, 0)
        rspeed = np.sqrt(rvx*rvx+rvy*rvy); np.fill_diagonal(rspeed, 0)
        ix1 = np.maximum(x1[:,None],x1[None,:]); iy1 = np.maximum(y1[:,None],y1[None,:])
        ix2 = np.minimum(x2[:,None],x2[None,:]); iy2 = np.minimum(y2[:,None],y2[None,:])
        inter = np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
        a_i = (x2-x1)*(y2-y1)
        union = a_i[:,None]+a_i[None,:]-inter
        iou = inter/np.maximum(union, 1e-12); np.fill_diagonal(iou, 0)
        k_eff = min(K, n-1)
        nn_idx = np.argpartition(dist, k_eff-1, axis=1)[:, :k_eff]
        ri = np.arange(n)[:, None]
        df.loc[positions, 'nn_dist_min'] = dist[ri, nn_idx].min(axis=1)
        df.loc[positions, 'nn_dist_mean'] = dist[ri, nn_idx].mean(axis=1)
        df.loc[positions, 'nn_closing_max'] = closing[ri, nn_idx].max(axis=1)
        df.loc[positions, 'nn_closing_mean'] = closing[ri, nn_idx].mean(axis=1)
        df.loc[positions, 'nn_iou_max'] = iou[ri, nn_idx].max(axis=1)
        df.loc[positions, 'nn_relative_speed_max'] = rspeed[ri, nn_idx].max(axis=1)
        if (gi+1) % 50000 == 0:
            print(f"    interaction [{gi+1}/{n_groups}]  {(time.time()-t1)/60:.1f}m")
    print(f"  interaction done ({(time.time()-t1)/60:.1f}m)")

    # ---- Temporal rolling ----
    print("\n  Computing temporal rolling features...")
    t2 = time.time()
    df = df.sort_values(['video','track_id','frame']).reset_index(drop=True)
    KEY = ['speed_norm','jerk_norm','area_change_rate','inverse_ttc','nn_closing_max','nn_dist_min']
    grp = df.groupby(['video','track_id'], sort=False)
    for f in KEY:
        df[f'{f}_w5_max']   = grp[f].transform(lambda s: s.rolling(5, min_periods=1).max())
        df[f'{f}_w10_max']  = grp[f].transform(lambda s: s.rolling(10, min_periods=1).max())
        df[f'{f}_w10_mean'] = grp[f].transform(lambda s: s.rolling(10, min_periods=1).mean())
        df[f'{f}_w10_std']  = grp[f].transform(lambda s: s.rolling(10, min_periods=1).std()).fillna(0)
    dd = grp['direction_rad'].diff().fillna(0)
    dd = np.arctan2(np.sin(dd), np.cos(dd))
    df['_dd_abs'] = dd.abs()
    df['direction_change_w10'] = df.groupby(['video','track_id'])['_dd_abs'].transform(
        lambda s: s.rolling(10, min_periods=1).sum())
    df = df.drop(columns=['_dd_abs'])
    df['speed_delta_w5'] = df['speed_norm'] - grp['speed_norm'].shift(5).fillna(0)
    print(f"  temporal done ({(time.time()-t2)/60:.1f}m)")

    df.to_parquet(CACHE_V3)
    print(f"Saved: {CACHE_V3}  shape={df.shape}")

# ============================================================
print("\n" + "="*72); print("Stage 3: Clean labels"); print("="*72)
# ============================================================
def iou_fn(a, b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1]); xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    ua=max(0,a[2]-a[0])*max(0,a[3]-a[1]); ub=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    den = ua+ub-inter
    return inter/den if den>0 else 0.0

CACHE_DET = OUT_DIR / "det_lookup.pkl"
import pickle
if CACHE_DET.exists():
    det_lookup = pickle.load(open(CACHE_DET, 'rb'))
    print(f"Loaded det_lookup ({len(det_lookup):,})")
else:
    det_lookup = {}
    for tp in track_files:
        vid = tp.stem
        for f in json.load(open(tp)):
            for d in f.get('detections', []):
                det_lookup[(vid, f['frame'], d['track_id'])] = d['bbox']
    pickle.dump(det_lookup, open(CACHE_DET, 'wb'))
    print(f"Built det_lookup: {len(det_lookup):,}")

gt_lookup = {}
for vid, ann in annotations.items():
    anom_s, anom_e = ann['anomaly_start'], ann['anomaly_end']
    for fl in ann['labels']:
        fid = fl['frame_id']
        if anom_s <= fid <= anom_e:
            boxes = [o['bbox'] for o in fl.get('objects', []) if o.get('bbox')]
            if boxes: gt_lookup[(vid, fid)] = boxes

clean_label = np.zeros(len(df), dtype=np.int32)
orig_labels = df['is_anomalous_orig'].values
case2 = case3 = 0
vid_arr = df['video'].values; fr_arr = df['frame'].values; tid_arr = df['track_id'].values

for ri in range(len(df)):
    vid = vid_arr[ri]; fid = int(fr_arr[ri]); tid = int(tid_arr[ri])
    if vid not in annotations:
        continue
    ann = annotations[vid]
    if not (ann['anomaly_start'] <= fid <= ann['anomaly_end']):
        continue
    gt_boxes = gt_lookup.get((vid, fid), [])
    dbox = det_lookup.get((vid, fid, tid))
    if dbox and gt_boxes:
        max_i = max(iou_fn(dbox, g) for g in gt_boxes)
        new_label = 1 if max_i >= 0.3 else 0
    else:
        new_label = 0
    clean_label[ri] = new_label
    if orig_labels[ri] == 0 and new_label == 1: case2 += 1
    elif orig_labels[ri] == 1 and new_label == 0: case3 += 1

df['is_anomalous'] = clean_label
print(f"Case 2 (0→1): {case2}")
print(f"Case 3 (1→0): {case3}")
print(f"Clean anomalous: {clean_label.sum():,} / {len(df):,} ({clean_label.mean():.2%})")

# ============================================================
print("\n" + "="*72); print("Stage 4: Classifier A v4-clean with residuals"); print("="*72)
# ============================================================
BASE = ['cx','cy','norm_x','norm_bottom_y','area_norm','aspect_ratio',
        'area_change_rate','vx_norm','vy_norm','speed_norm','direction_rad',
        'v4_flow_decoupled_score','ax_norm','ay_norm','jerk_norm',
        'track_age','track_stability','yolo_class']
NEW_V2 = ['nn_dist_min','nn_dist_mean','nn_closing_max','nn_closing_mean',
          'nn_iou_max','nn_relative_speed_max','n_others_in_frame',
          'ttc_frames','inverse_ttc']
TEMPORAL = [c for c in df.columns
            if any(c.endswith(s) for s in ['_w5_max','_w10_max','_w10_mean','_w10_std'])
            or c in ['direction_change_w10','speed_delta_w5']]

print("Computing residuals...")
state_cols = ['cx','cy','area_norm','aspect_ratio','vx_norm','vy_norm']
target_cols = ['cx','cy','area_norm','aspect_ratio']
for lag in range(1, 6):
    for c in state_cols:
        df[f'{c}_lag{lag}'] = df.groupby(['video','track_id'])[c].shift(lag)
hist_cols = [f'{c}_lag{lag}' for lag in range(1,6) for c in state_cols]
valid = df[hist_cols + target_cols].notna().all(axis=1)
residual_cols = [f'residual_{t}' for t in target_cols]
for c in residual_cols: df[c] = 0.0

video_has_anom = df.groupby('video')['is_anomalous'].max().to_dict()
vid_list = sorted(df['video'].unique())
vid_strat = [video_has_anom[v] for v in vid_list]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fi, (tr_vi, te_vi) in enumerate(skf.split(vid_list, vid_strat)):
    tr_vids = {vid_list[i] for i in tr_vi}; te_vids = {vid_list[i] for i in te_vi}
    tr_m = df['video'].isin(tr_vids).values
    te_m = df['video'].isin(te_vids).values
    fit_m = tr_m & valid.values & (df['is_anomalous'].values == 0)
    X_fit = df.loc[fit_m, hist_cols].values.astype(np.float32)
    for target in target_cols:
        y_fit = df.loc[fit_m, target].values.astype(np.float32)
        reg = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, max_depth=5,
                                num_leaves=31, min_child_samples=100,
                                n_jobs=-1, random_state=42, verbose=-1)
        reg.fit(X_fit, y_fit)
        te_v = te_m & valid.values
        if te_v.sum() > 0:
            pred = reg.predict(df.loc[te_v, hist_cols].values.astype(np.float32))
            actual = df.loc[te_v, target].values.astype(np.float32)
            df.loc[te_v, f'residual_{target}'] = np.abs(pred - actual)
    print(f"  Residual fold {fi+1} done")

FEATURES = BASE + NEW_V2 + TEMPORAL + residual_cols
print(f"Classifier A features: {len(FEATURES)}")
X_df = df[FEATURES].copy()
y = df['is_anomalous'].values.astype(np.int32)

aucs = []
probs_store = np.zeros(len(df))
for fi, (tr_vi, te_vi) in enumerate(skf.split(vid_list, vid_strat)):
    tr_vids = {vid_list[i] for i in tr_vi}; te_vids = {vid_list[i] for i in te_vi}
    tr_m = df['video'].isin(tr_vids).values; te_m = df['video'].isin(te_vids).values
    pw = (y[tr_m]==0).sum() / max(1, (y[tr_m]==1).sum())
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                           num_leaves=31, min_child_samples=50,
                           scale_pos_weight=pw, n_jobs=-1, random_state=42, verbose=-1)
    m.fit(X_df.iloc[tr_m], y[tr_m])
    p = m.predict_proba(X_df.iloc[te_m])[:, 1]
    probs_store[te_m] = p
    auc = roc_auc_score(y[te_m], p); aucs.append(auc)
    print(f"  Classifier A fold {fi+1}: AUC={auc:.4f}")
print(f"  v4-clean mean AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  (200-video baseline was 0.8869)")

pred_clean = pd.DataFrame({'video': df['video'].values, 'frame': df['frame'].values,
                           'track_id': df['track_id'].values,
                           'prob_anomaly': probs_store, 'true_label_clean': y})
pred_clean.to_parquet(OUT_DIR / "classifier_a_v4_clean_predictions_full.parquet")

# ============================================================
print("\n" + "="*72); print("Stage 5: Stage-2 frame classifier"); print("="*72)
# ============================================================
track_ages = {}
tracks_bf = {}
for tp in track_files:
    vid = tp.stem
    tracks_bf[vid] = {}
    tid_first = {}
    for f in json.load(open(tp)):
        fid = f['frame']
        tracks_bf[vid][fid] = f.get('detections', [])
        for d in f.get('detections', []):
            tid = d['track_id']
            if tid not in tid_first: tid_first[tid] = fid
            track_ages.setdefault((vid, tid), {})[fid] = fid - tid_first[tid]

pred_lookup = {(r.video, r.frame, r.track_id): r.prob_anomaly
               for r in pred_clean.itertuples(index=False)}

frame_rows = []
for vid in eval_videos:
    if vid not in annotations: continue
    ann = annotations[vid]
    for fid, dets in tracks_bf.get(vid, {}).items():
        scores = []; ages = []
        for d in dets:
            k = (vid, fid, d['track_id'])
            if k in pred_lookup:
                scores.append(pred_lookup[k])
                ages.append(track_ages.get((vid, d['track_id']), {}).get(fid, 0))
        if not scores: continue
        s = np.array(scores); a = np.array(ages)
        sorted_s = np.sort(s)[::-1]; K = len(s)
        frame_rows.append({
            'video': vid, 'frame': fid,
            'label': 1 if ann['anomaly_start']<=fid<=ann['anomaly_end'] else 0,
            'max_score': sorted_s[0], 'mean_score': s.mean(),
            'std_score': s.std() if K>1 else 0,
            'top3_mean': sorted_s[:min(3,K)].mean(),
            'top3_max_min_diff': sorted_s[0]-sorted_s[min(2,K-1)],
            'pct90': np.percentile(s, 90),
            'count_gt_03': (s>0.3).sum(), 'count_gt_05': (s>0.5).sum(),
            'count_gt_07': (s>0.7).sum(), 'num_objects': K,
            'max_minus_mean': sorted_s[0]-s.mean(),
            'max_over_mean': sorted_s[0]/(s.mean()+1e-6),
            'max_track_age': a.max() if len(a)>0 else 0,
            'max_score_track_age': a[np.argmax(s)] if len(a)>0 else 0,
        })

fdf = pd.DataFrame(frame_rows).sort_values(['video','frame']).reset_index(drop=True)
for w, span in [('ema_3',3),('ema_5',5),('ema_10',10)]:
    fdf[f'max_{w}'] = fdf.groupby('video')['max_score'].transform(lambda s: s.ewm(span=span).mean())
fdf['mean_ema_5'] = fdf.groupby('video')['mean_score'].transform(lambda s: s.ewm(span=5).mean())
fdf['max_diff'] = fdf.groupby('video')['max_score'].diff().fillna(0)
fdf['max_rolling3_max'] = fdf.groupby('video')['max_score'].transform(lambda s: s.rolling(3, min_periods=1).max())
fdf['max_rolling5_max'] = fdf.groupby('video')['max_score'].transform(lambda s: s.rolling(5, min_periods=1).max())
fdf['max_rolling5_mean'] = fdf.groupby('video')['max_score'].transform(lambda s: s.rolling(5, min_periods=1).mean())
print(f"Frame dataset: {len(fdf):,}  ({fdf['label'].mean():.1%} anomalous)")

STAGE2 = ['max_score','mean_score','std_score','top3_mean','top3_max_min_diff',
          'pct90','count_gt_03','count_gt_05','count_gt_07','num_objects',
          'max_minus_mean','max_over_mean','max_track_age','max_score_track_age',
          'max_ema_3','max_ema_5','max_ema_10','mean_ema_5','max_diff',
          'max_rolling3_max','max_rolling5_max','max_rolling5_mean']
X_f = fdf[STAGE2].values.astype(np.float32)
y_f = fdf['label'].values.astype(np.int32)
fvid_list = sorted(fdf['video'].unique())
fvid_strat = [int(fdf[fdf['video']==v]['label'].max()) for v in fvid_list]

stage2_probs = np.zeros(len(fdf))
stage2_aucs = []
for fi, (tr_vi, te_vi) in enumerate(skf.split(fvid_list, fvid_strat)):
    tr_vids = {fvid_list[i] for i in tr_vi}; te_vids = {fvid_list[i] for i in te_vi}
    tr_m = fdf['video'].isin(tr_vids).values; te_m = fdf['video'].isin(te_vids).values
    pw = (y_f[tr_m]==0).sum() / max(1, (y_f[tr_m]==1).sum())
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                           num_leaves=31, min_child_samples=50,
                           scale_pos_weight=pw, n_jobs=-1, random_state=42, verbose=-1)
    m.fit(X_f[tr_m], y_f[tr_m])
    p = m.predict_proba(X_f[te_m])[:, 1]
    stage2_probs[te_m] = p
    auc = roc_auc_score(y_f[te_m], p); stage2_aucs.append(auc)
    print(f"  Stage-2 fold {fi+1}: AUC={auc:.4f}")
print(f"  Stage-2 mean: {np.mean(stage2_aucs):.4f} ± {np.std(stage2_aucs):.4f}")
print(f"  (200-video baseline was 0.7634)")
fdf['stage2_prob'] = stage2_probs

# ============================================================
print("\n" + "="*72); print("Stage 6: STAUC on full DoTA"); print("="*72)
# ============================================================
H_IMG, W_IMG = 256, 256

def bmap(bbs, sc, img_size=(1280,720)):
    bbs = copy.deepcopy(bbs)
    if isinstance(bbs, list): bbs = torch.tensor(bbs, dtype=torch.float32)
    sm = torch.zeros(H_IMG, W_IMG)
    if len(bbs)==0: return sm
    if bbs.max()>1:
        bbs[:,[0,2]] /= img_size[0]; bbs[:,[1,3]] /= img_size[1]
        bbs[:,[0,2]] *= W_IMG; bbs[:,[1,3]] *= H_IMG
    bbs = bbs.type(torch.long)
    bbs[:,[0,2]] = torch.clamp(bbs[:,[0,2]], 0, W_IMG); bbs[:,[1,3]] = torch.clamp(bbs[:,[1,3]], 0, H_IMG)
    for bb, s in zip(bbs, sc):
        wb=(bb[2]-bb[0]).item(); hb=(bb[3]-bb[1]).item()
        if wb<=0 or hb<=0: continue
        sig = torch.tensor([wb,hb], dtype=torch.float32)
        xl = torch.arange(0,wb,1,dtype=torch.float32)
        yl = torch.arange(0,hb,1,dtype=torch.float32)[:,None]
        x0=wb//2; y0=hb//2
        g = torch.exp(-(((xl-x0)**2)/sig[0] + ((yl-y0)**2)/sig[1])/2)
        sm[bb[1]:bb[3], bb[0]:bb[2]] += g * float(s)
    return sm

def tarr(lab, gt, ob, sc):
    if lab==0: return 0.0
    if len(gt)==0: return 1.0
    if len(ob)==0 or len(sc)==0: return 1.0
    sm = bmap(ob, sc)
    g = torch.tensor(copy.deepcopy(gt), dtype=torch.float32)
    if g.max()>1:
        g[:,[0,2]] /= 1280; g[:,[1,3]] /= 720
        g[:,[0,2]] *= W_IMG; g[:,[1,3]] *= H_IMG
    g = g.type(torch.long)
    g[:,[0,2]] = torch.clamp(g[:,[0,2]], 0, W_IMG); g[:,[1,3]] = torch.clamp(g[:,[1,3]], 0, H_IMG)
    mask = torch.zeros(H_IMG, W_IMG)
    for bb in g: mask[bb[1]:bb[3], bb[0]:bb[2]] = 1
    K_m = int(mask.sum().item())
    if K_m==0: return 1.0
    vals, ii = torch.topk(sm.view(-1), k=min(K_m, H_IMG*W_IMG))
    hc = ii // W_IMG; wc = ii % W_IMG
    hits = mask[hc, wc] == 1
    return float(vals[hits].sum().item() / (vals.sum().item() + 1e-6))

def cs(lbl, scr, tr):
    lbl=np.asarray(lbl); scr=np.asarray(scr); tr=np.asarray(tr)
    d = np.argsort(scr, kind='mergesort')[::-1]
    ys = (lbl[d]==1).astype(float); ws = tr[d]
    dst = np.where(np.diff(scr[d]))[0]; ti = np.r_[dst, len(ys)-1]
    tps = stable_cumsum(ys*ws)[ti]; pos = stable_cumsum(ys)[ti]; fps = stable_cumsum(1-ys)[ti]
    tps=np.r_[0,tps]; fps=np.r_[0,fps]; pos=np.r_[0,pos]
    if fps[-1]<=0 or pos[-1]<=0: return float('nan'), float('nan')
    return sk_metrics.auc(fps/fps[-1], tps/pos[-1]), sk_metrics.auc(fps/fps[-1], pos/pos[-1])

def gt_pf(ann):
    return {fl['frame_id']: ([o['bbox'] for o in fl['objects']]
                              if ann['anomaly_start']<=fl['frame_id']<=ann['anomaly_end'] else [])
            for fl in ann['labels']}

frame_score_lookup = {(r.video, r.frame): r.stage2_prob for r in fdf.itertuples(index=False)}
obj_lookup = {}
for vid in eval_videos:
    if vid not in tracks_bf: continue
    for fid, dets in tracks_bf[vid].items():
        pairs = []
        for d in dets:
            k = (vid, fid, d['track_id'])
            if k in pred_lookup:
                pairs.append((d['bbox'], pred_lookup[k]))
        if pairs: obj_lookup[(vid, fid)] = pairs

t_stauc = time.time()
all_y, all_s, all_t = [], [], []
for vi, vid in enumerate(eval_videos):
    if vid not in tracks_bf: continue
    ann = annotations[vid]; gt = gt_pf(ann)
    for fid in tracks_bf[vid]:
        gb = gt.get(fid, [])
        lab = 1 if ann['anomaly_start']<=fid<=ann['anomaly_end'] else 0
        pairs = obj_lookup.get((vid, fid), [])
        if not pairs:
            all_y.append(lab); all_s.append(0.0)
            all_t.append(0.0 if lab==0 else (1.0 if len(gb)==0 else 0.0)); continue
        bx = [p[0] for p in pairs]; sc = [p[1] for p in pairs]
        t = tarr(lab, gb, bx, sc)
        fs = frame_score_lookup.get((vid, fid), float(np.max(sc)))
        all_y.append(lab); all_s.append(fs); all_t.append(t)
    if (vi + 1) % 500 == 0:
        print(f"  STAUC [{vi+1}/{len(eval_videos)}]  {(time.time()-t_stauc)/60:.1f}m")

stauc, auc_final = cs(all_y, all_s, all_t)
print(f"\nFULL DoTA ({len(eval_videos)} videos):")
print(f"  Frame AUC: {auc_final:.4f}")
print(f"  STAUC:     {stauc:.4f}")
print(f"  Gap:       {auc_final-stauc:+.4f}")

# ============================================================
print("\n" + "="*72); print("FINAL PROGRESSION"); print("="*72)
# ============================================================
print(f"{'Method':<50} {'fAUC':>8} {'STAUC':>8}")
print("-"*66)
print(f"{'200-video Stage-2 (v4-clean)':<50} {0.6891:>8.4f} {0.4895:>8.4f}")
print(f"{'FULL DoTA Stage-2 (THIS RUN)':<50} {auc_final:>8.4f} {stauc:>8.4f}")
print(f"\nYao FOL-Ensemble (unsup, 4677): fAUC=0.7300  STAUC=0.4850")
print(f"Yao FOL-STD best (unsup): fAUC=0.6970  STAUC=0.4370")
print(f"\nDelta vs Yao FOL-Ensemble: fAUC {auc_final-0.73:+.4f}  STAUC {stauc-0.485:+.4f}")

json.dump({
    'full_dota_n_videos': len(eval_videos),
    'classifier_a_cv_auc': float(np.mean(aucs)),
    'stage2_cv_auc': float(np.mean(stage2_aucs)),
    'final_frame_auc': float(auc_final),
    'final_stauc': float(stauc),
    'case2_fixes': int(case2), 'case3_fixes': int(case3),
    'n_anomalous_frame_samples': int(clean_label.sum()),
    'total_frame_samples': int(len(df)),
}, open(OUT_DIR / "session2_final_results.json", 'w'), indent=2)
print(f"\nSaved: {OUT_DIR / 'session2_final_results.json'}")
print("\nRun 'Save Version → Save & Run All (Commit)' after this completes to persist results.")
print("DONE")